# Import Lib

In [234]:
!pip install torcheval

In [235]:
import os
import joblib
import torch
import torch.nn as nn
import transformers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from glob import glob

from sklearn import preprocessing
from sklearn import model_selection

from tqdm import tqdm
from transformers import AdamW
from transformers import get_linear_schedule_with_warmup
from torcheval.metrics.functional import multiclass_f1_score

warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.optimization")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:

def read_conll_file(file_path):
    sentences = []
    labels = []
    with open(file_path, "r", encoding="utf-8") as f:
        current_words = []
        current_labels = []
        for line in f:
            line = line.strip()
            if not line:
                    sentences.append(current_words)
                    labels.append(current_labels)
                    current_words = []
                    current_labels = []
            else:
                splits = line.split()
                word = splits[0]
                tag = splits[-1] 
                current_words.append(word)
                current_labels.append(tag)
        
        if current_words:
            sentences.append(current_words)
            labels.append(current_labels)
            
    return sentences, labels

In [ ]:

train_words, train_words_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/word/train_word.conll')
dev_words, dev_words_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/word/dev_word.conll')
test_words, test_words_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/word/test_word.conll')


unique_words_tags = set(tag for doc in train_words_tags for tag in doc)
label_words_list = sorted(list(unique_words_tags))
label2id_words = {label: i for i, label in enumerate(label_words_list)}
id2label_words = {i: label for i, label in enumerate(label_words_list)}

print("Danh sách nhãn:", label_words_list)

Danh sách nhãn: ['B-AGE', 'B-DATE', 'B-GENDER', 'B-JOB', 'B-LOCATION', 'B-NAME', 'B-ORGANIZATION', 'B-PATIENT_ID', 'B-SYMPTOM_AND_DISEASE', 'B-TRANSPORTATION', 'I-AGE', 'I-DATE', 'I-JOB', 'I-LOCATION', 'I-NAME', 'I-ORGANIZATION', 'I-PATIENT_ID', 'I-SYMPTOM_AND_DISEASE', 'I-TRANSPORTATION', 'O']


In [ ]:

train_syllable, train_syllable_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/syllable/train_syllable.conll')
dev_syllable, dev_syllable_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/syllable/dev_syllable.conll')
test_syllable, test_syllable_tags = read_conll_file('/kaggle/input/datasets/sonbui13/phoner-covid19/syllable/test_syllable.conll')

 
unique_syllable_tags = set(tag for doc in train_syllable_tags for tag in doc)
label_syllable_list = sorted(list(unique_syllable_tags))
label2id_syllable = {label: i for i, label in enumerate(label_syllable_list)}
id2label_syllable = {i: label for i, label in enumerate(label_syllable_list)}

print("Danh sách nhãn:", label_syllable_list)

Danh sách nhãn: ['B-AGE', 'B-DATE', 'B-GENDER', 'B-JOB', 'B-LOCATION', 'B-NAME', 'B-ORGANIZATION', 'B-PATIENT_ID', 'B-SYMPTOM_AND_DISEASE', 'B-TRANSPORTATION', 'I-AGE', 'I-DATE', 'I-GENDER', 'I-JOB', 'I-LOCATION', 'I-NAME', 'I-ORGANIZATION', 'I-PATIENT_ID', 'I-SYMPTOM_AND_DISEASE', 'I-TRANSPORTATION', 'O']


In [240]:
from collections import Counter

def build_vocab(sentences, min_freq=1):
    counter = Counter(word for sent in sentences for word in sent)

    word2id = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            word2id[word] = len(word2id)

    return word2id


def build_char_vocab(sentences):
    chars = set(c for sent in sentences for word in sent for c in word)

    char2id = {"<PAD>": 0, "<UNK>": 1}
    for c in chars:
        char2id[c] = len(char2id)

    return char2id


def build_label_vocab(tags):
    unique = sorted(set(t for doc in tags for t in doc))

    if "O" in unique:
        unique.remove("O")
        unique = ["O"] + unique

    label2id = {l: i for i, l in enumerate(unique)}
    id2label = {i: l for l, i in label2id.items()}

    return label2id, id2label

In [241]:
import torch
from torch.utils.data import Dataset

class NERDataset(Dataset):
    def __init__(self, texts, tags, word2id, char2id, label2id, max_len=80):
        self.texts = texts
        self.tags = tags
        self.word2id = word2id
        self.char2id = char2id
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        sent = self.texts[idx]
        tag = self.tags[idx]

        word_ids = [self.word2id.get(w, self.word2id["<UNK>"]) for w in sent]
        char_ids = [[self.char2id.get(c, self.char2id["<UNK>"]) for c in word] for word in sent]
        label_ids = [self.label2id[t] for t in tag]

        word_ids = word_ids[:self.max_len]
        char_ids = char_ids[:self.max_len]
        label_ids = label_ids[:self.max_len]

        return {
            "word_ids": torch.tensor(word_ids, dtype=torch.long),
            "char_ids": char_ids,
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }

In [242]:
def collate_fn(batch):
    batch_size = len(batch)
    max_seq_len = max(len(x["word_ids"]) for x in batch)
    max_char_len = max(max(len(c) for c in x["char_ids"]) for x in batch)

    word_ids = torch.zeros(batch_size, max_seq_len, dtype=torch.long)
    char_ids = torch.zeros(batch_size, max_seq_len, max_char_len, dtype=torch.long)
    labels = torch.zeros(batch_size, max_seq_len, dtype=torch.long)
    mask = torch.zeros(batch_size, max_seq_len, dtype=torch.bool)

    for i, item in enumerate(batch):
        seq_len = len(item["word_ids"])
        word_ids[i, :seq_len] = item["word_ids"]
        labels[i, :seq_len] = item["labels"]
        mask[i, :seq_len] = True

        for j, chars in enumerate(item["char_ids"]):
            char_ids[i, j, :len(chars)] = torch.tensor(chars)

    return {
        "word_ids": word_ids,
        "char_ids": char_ids,
        "labels": labels,
        "mask": mask
    }

In [243]:
!pip install -q gensim

In [244]:
from gensim.models import KeyedVectors
import numpy as np

WORD2VEC_PATH = "/kaggle/input/datasets/duynm619/word2vecvn-trained-model/wiki.vi.model.bin"
WORD2VEC_BINARY = True   

w2v = KeyedVectors.load_word2vec_format(
    WORD2VEC_PATH,
    binary=WORD2VEC_BINARY
)

def build_embedding_word2vec(word2id, w2v_model):
    dim = w2v_model.vector_size  
    
    embedding = np.random.uniform(-0.25, 0.25, (len(word2id), dim)).astype(np.float32)
    embedding[word2id["<PAD>"]] = np.zeros(dim, dtype=np.float32)

    for word, idx in word2id.items():
        if word in w2v_model:
            embedding[idx] = w2v_model[word]

    return embedding

In [245]:
!pip install pytorch-crf

In [246]:
from torchcrf import CRF
import torch.nn as nn
import torch

class BiLSTM_CNN_CRF(nn.Module):
    def __init__(self, vocab_size, char_size, tagset_size, embedding_matrix):
        super().__init__()

        embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float)

        self.word_emb = nn.Embedding.from_pretrained(
            embedding_tensor,
            freeze=True,
            padding_idx=0
        )

        word_dim = embedding_tensor.shape[1]

        self.char_emb = nn.Embedding(char_size, 50, padding_idx=0)
        self.char_cnn = nn.Conv1d(50, 30, kernel_size=3, padding=1)

        self.lstm = nn.LSTM(
            input_size=word_dim + 30,
            hidden_size=200,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.25
        )

        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(400, tagset_size)

        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, word_ids, char_ids, tags=None, mask=None):
        word_emb = self.word_emb(word_ids)

        B, T, C = char_ids.shape
        char_ids = char_ids.view(B * T, C)

        char_emb = self.char_emb(char_ids).transpose(1, 2)
        char_cnn = torch.relu(self.char_cnn(char_emb))
        char_pool = torch.max(char_cnn, dim=2)[0]
        char_pool = char_pool.view(B, T, -1)

        x = torch.cat([word_emb, char_pool], dim=-1)

        x, _ = self.lstm(x)
        x = self.dropout(x)

        emissions = self.fc(x)

        if tags is not None:
            return -self.crf(emissions, tags, mask=mask.bool())

        return self.crf.decode(emissions, mask=mask.bool())

In [247]:
pip install -q seqeval --upgrade --no-cache-dir

Note: you may need to restart the kernel to use updated packages.


In [248]:
from seqeval.metrics import f1_score, precision_score, recall_score

def evaluate(model, dataloader, id2label, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            word_ids = batch["word_ids"].to(device)
            char_ids = batch["char_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["mask"].to(device)

            preds = model(word_ids, char_ids, mask=mask)

            for i in range(len(preds)):
                pred_seq = preds[i]
                true_seq = labels[i][mask[i]].cpu().numpy()

                all_preds.append([id2label[p] for p in pred_seq])
                all_labels.append([id2label[t] for t in true_seq])

    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return precision, recall, f1

In [249]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch

word2id_words = build_vocab(train_words)
char2id_words = build_char_vocab(train_words)
label2id_words, id2label_words = build_label_vocab(train_words_tags)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_matrix_words = build_embedding_word2vec(word2id_words, w2v)

train_words_dataset = NERDataset(train_words, train_words_tags, word2id_words, char2id_words, label2id_words)
dev_words_dataset = NERDataset(dev_words, dev_words_tags, word2id_words, char2id_words, label2id_words)

train_words_loader = DataLoader(train_words_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
dev_words_loader = DataLoader(dev_words_dataset, batch_size=16, collate_fn=collate_fn)

model_words = BiLSTM_CNN_CRF(len(word2id_words), len(char2id_words), len(label2id_words), embedding_matrix_words).to(device)

optimizer_words = optim.Adam(model_words.parameters(), lr=0.001)

EPOCHS = 30
best_f1 = 0
patience = 0

for epoch in range(EPOCHS):
    model_words.train()
    total_loss = 0

    for batch in train_words_loader:
        word_ids = batch["word_ids"].to(device)
        char_ids = batch["char_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        optimizer_words.zero_grad()
        loss = model_words(word_ids, char_ids, labels, mask)
        loss.backward()
        optimizer_words.step()

        total_loss += loss.item()

    p, r, f1 = evaluate(model_words, dev_words_loader, id2label_words, device)

    print(f"Epoch {epoch+1} | Loss {total_loss:.4f} | F1 {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        patience = 0
        torch.save(model_words.state_dict(), "best_words_model.pt")
    else:
        patience += 1

    if patience >= 5:
        print("Early stopping")
        break

Epoch 1 | Loss 47841.3673 | F1 0.8249
Epoch 2 | Loss 11551.8249 | F1 0.8848
Epoch 3 | Loss 6734.9065 | F1 0.9007
Epoch 4 | Loss 4737.4835 | F1 0.9124
Epoch 5 | Loss 3317.6548 | F1 0.9079
Epoch 6 | Loss 2792.1956 | F1 0.9177
Epoch 7 | Loss 2009.7410 | F1 0.8823
Epoch 8 | Loss 1510.8226 | F1 0.9111
Epoch 9 | Loss 1179.0044 | F1 0.9184
Epoch 10 | Loss 1145.0320 | F1 0.9143
Epoch 11 | Loss 1092.2238 | F1 0.9164
Epoch 12 | Loss 956.1062 | F1 0.9205
Epoch 13 | Loss 700.9366 | F1 0.9123
Epoch 14 | Loss 762.1211 | F1 0.9196
Epoch 15 | Loss 615.7374 | F1 0.9128
Epoch 16 | Loss 698.6663 | F1 0.9135
Epoch 17 | Loss 387.5416 | F1 0.9185
Early stopping


In [250]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch

word2id_syllable = build_vocab(train_syllable)
char2id_syllable = build_char_vocab(train_syllable)
label2id_syllable, id2label_syllable = build_label_vocab(train_syllable_tags)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_matrix_syllable = build_embedding_word2vec(word2id_syllable, w2v)

train_syllable_dataset = NERDataset(train_syllable, train_syllable_tags, word2id_syllable, char2id_syllable, label2id_syllable)
dev_syllable_dataset = NERDataset(dev_syllable, dev_syllable_tags, word2id_syllable, char2id_syllable, label2id_syllable)

train_syllable_loader = DataLoader(train_syllable_dataset, batch_size=36, shuffle=True, collate_fn=collate_fn)
dev_syllable_loader = DataLoader(dev_syllable_dataset, batch_size=36, collate_fn=collate_fn)

model_syllable = BiLSTM_CNN_CRF(len(word2id_syllable), len(char2id_syllable), len(label2id_syllable), embedding_matrix_syllable).to(device)

optimizer_syllable = optim.Adam(model_syllable.parameters(), lr=0.001)

EPOCHS = 30
best_f1 = 0
patience = 0

for epoch in range(EPOCHS):
    model_syllable.train()
    total_loss = 0

    for batch in train_syllable_loader:
        word_ids = batch["word_ids"].to(device)
        char_ids = batch["char_ids"].to(device)
        labels = batch["labels"].to(device)
        mask = batch["mask"].to(device)

        optimizer_syllable.zero_grad()
        loss = model_syllable(word_ids, char_ids, labels, mask)
        loss.backward()
        optimizer_syllable.step()

        total_loss += loss.item()

    p, r, f1 = evaluate(model_syllable, dev_syllable_loader, id2label_syllable, device)

    print(f"Epoch {epoch+1} | Loss {total_loss:.4f} | F1 {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        patience = 0
        torch.save(model_syllable.state_dict(), "best_syllable_model.pt")
    else:
        patience += 1

    if patience >= 5:
        print("Early stopping")
        break

Epoch 1 | Loss 91683.6580 | F1 0.8020
Epoch 2 | Loss 16701.4799 | F1 0.8601
Epoch 3 | Loss 9837.7523 | F1 0.8779
Epoch 4 | Loss 7022.0431 | F1 0.8993
Epoch 5 | Loss 5416.9141 | F1 0.9017
Epoch 6 | Loss 3680.7498 | F1 0.9050
Epoch 7 | Loss 3199.0893 | F1 0.9057
Epoch 8 | Loss 2434.9157 | F1 0.9122
Epoch 9 | Loss 1809.2382 | F1 0.9080
Epoch 10 | Loss 1595.4124 | F1 0.9126
Epoch 11 | Loss 1308.7260 | F1 0.9147
Epoch 12 | Loss 872.8749 | F1 0.9129
Epoch 13 | Loss 905.4033 | F1 0.9108
Epoch 14 | Loss 721.0616 | F1 0.9094
Epoch 15 | Loss 563.1708 | F1 0.9138
Epoch 16 | Loss 498.9005 | F1 0.9118
Early stopping


In [251]:
precision_words, recall_words, f1_words = evaluate(model_words, dev_words_loader, id2label_words, device)

print("\nWords - Key Metrics (Micro Average):")
print(f"- Micro F1-Score: {f1_words:.4f}")
print(f"- Micro Precision: {precision_words:.4f}")
print(f"- Micro Recall: {recall_words:.4f}")


Words - Key Metrics (Micro Average):
- Micro F1-Score: 0.9185
- Micro Precision: 0.9201
- Micro Recall: 0.9169


In [252]:
precision_syllable, recall_syllable, f1_syllable = evaluate(model_syllable, dev_syllable_loader, id2label_syllable, device)

print("\nSyllable - Key Metrics (Micro Average):")
print(f"- Micro F1-Score: {f1_syllable:.4f}")
print(f"- Micro Precision: {precision_syllable:.4f}")
print(f"- Micro Recall: {recall_syllable:.4f}")


Syllable - Key Metrics (Micro Average):
- Micro F1-Score: 0.9118
- Micro Precision: 0.9070
- Micro Recall: 0.9166


In [253]:
from seqeval.metrics import classification_report

def entity_report(model, dataloader, id2label, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            word_ids = batch["word_ids"].to(device)
            char_ids = batch["char_ids"].to(device)
            labels = batch["labels"].to(device)
            mask = batch["mask"].to(device)

            preds = model(word_ids, char_ids, mask=mask)

            for i in range(len(preds)):
                pred_seq = preds[i]
                true_seq = labels[i][mask[i]].cpu().numpy()

                all_preds.append([id2label[p] for p in pred_seq])
                all_labels.append([id2label[t] for t in true_seq])

    print(classification_report(all_labels, all_preds, digits=4))

In [254]:
print("Words - Report:")
entity_report(model_words, dev_words_loader, id2label_words, device)

Words - Report:
                     precision    recall  f1-score   support

                AGE     0.9553    0.9607    0.9580       356
               DATE     0.9609    0.9791    0.9699      1103
             GENDER     0.9487    0.9453    0.9470       274
                JOB     0.8163    0.6107    0.6987       131
           LOCATION     0.9177    0.9322    0.9249      2727
               NAME     0.8902    0.8280    0.8579       186
       ORGANIZATION     0.8725    0.7949    0.8319       551
         PATIENT_ID     0.9573    0.9890    0.9729      1269
SYMPTOM_AND_DISEASE     0.8201    0.7794    0.7992       766
     TRANSPORTATION     0.9655    0.9655    0.9655        87

          micro avg     0.9201    0.9169    0.9185      7450
          macro avg     0.9104    0.8785    0.8926      7450
       weighted avg     0.9185    0.9169    0.9171      7450



In [255]:
print("Syllable - Report:")
entity_report(model_syllable, dev_syllable_loader, id2label_syllable, device)

Syllable - Report:
                     precision    recall  f1-score   support

                AGE     0.9413    0.9656    0.9533       349
               DATE     0.9593    0.9837    0.9713      1102
             GENDER     0.9403    0.9299    0.9351       271
                JOB     0.8529    0.6744    0.7532       129
           LOCATION     0.8978    0.9213    0.9094      2707
               NAME     0.8674    0.8441    0.8556       186
       ORGANIZATION     0.8426    0.8303    0.8364       548
         PATIENT_ID     0.9672    0.9786    0.9729      1264
SYMPTOM_AND_DISEASE     0.7947    0.7906    0.7927       764
     TRANSPORTATION     0.9213    0.9425    0.9318        87

          micro avg     0.9070    0.9166    0.9118      7407
          macro avg     0.8985    0.8861    0.8912      7407
       weighted avg     0.9064    0.9166    0.9112      7407



In [256]:
from huggingface_hub import create_repo

repo_id = "ttnuizrachel/BiLSTM-CNN-CRF"

create_repo(repo_id, exist_ok=True)

RepoUrl('https://huggingface.co/ttnuizrachel/BiLSTM-CNN-CRF', endpoint='https://huggingface.co', repo_type='model', repo_id='ttnuizrachel/BiLSTM-CNN-CRF')

In [266]:
README_text = """
---
language:
  - vi.
tags:
  - ner.
  - medical.
  - covid-19.
  - bilstm-cnn-crf.
  - pytorch.
  - token-classification.
metrics:
  - precision.
  - recall.
  - f1.
---

# 🇻🇳 BiLSTM-CNN-CRF for Vietnamese COVID-19 NER

## 📌 Overview

This repository provides a **Named Entity Recognition (NER)** model for Vietnamese text in the **COVID-19 medical domain**.

The model is built using a hybrid deep learning architecture: **BiLSTM-CNN-CRF**, which captures both contextual and morphological features.

It is trained on the **PhoNER_COVID19 dataset** for extracting structured medical information from clinical and COVID-related texts.

---

## 🧩 Motivation

Vietnamese NER suffers from ambiguity in word segmentation. To address this, we explore two granularities:

- **Word-level representation**: captures semantic context.
- **Syllable-level representation**: better handles Vietnamese tokenization ambiguity and out-of-vocabulary words.

---

## 🧠 Model Architecture

We propose two variants of the BiLSTM-CNN-CRF model:

### 🔹 1. Word-level Model
- Pre-trained FastText / Word2Vec embeddings (300-dim).
- Character-level CNN (50-dim).
- BiLSTM encoder (hidden=200, 2 layers).
- CRF decoder.

### 🔹 2. Syllable-level Model
- Syllable-based tokenization input.
- Word embeddings trained on syllable vocabulary.
- Character-level CNN (50-dim).
- BiLSTM encoder (hidden=200, 2 layers).
- CRF decoder.

---

## 🏷️ Named Entities

| Entity              | Description        |
|---------------------|--------------------|
| AGE                 | Tuổi               |
| DATE                | Ngày tháng         |
| GENDER              | Giới tính          |
| JOB                 | Nghề nghiệp        |
| LOCATION            | Địa điểm           |
| NAME                | Tên người          |
| ORGANIZATION        | Tổ chức            |
| PATIENT_ID          | Mã bệnh nhân       |
| SYMPTOM_AND_DISEASE | Triệu chứng & Bệnh |
| TRANSPORTATION      | Phương tiện        |

---

## ⚙️ Training Configuration

| Parameter      |  Value  |
|----------------|---------|
| Word Embedding | 300     |
| Char Embedding | 50      |
| LSTM Hidden    | 200     |
| Dropout        | 0.25    |
| Optimizer      | Adam    |
| Learning Rate  | 0.001   |
| Batch Size     | 36      |
| Epochs         | 30      |
| Early Stopping | 19      |

---

## 📊 Evaluation Results

### 🧠 Word-level Model

#### Overall Performance

- **Micro Precision:** 0.9201    
- **Micro Recall:** 0.9169  
- **Micro F1-score:** 0.9185    

- **Macro F1-score:** 0.8926  
- **Weighted F1-score:** 0.9171           

### 🧠 Syllable-level Model

#### Overall Performance

- **Micro Precision:** 0.9070    
- **Micro Recall:** 0.9166
- **Micro F1-score:** 0.9118 

- **Macro F1-score:** 0.8912
- **Weighted F1-score:** 0.9112 


---

## 📈 Detailed Results by (Word Model)

| Entity              | Precision | Recall | F1-score | Support |
|---------------------|-----------|--------|----------|---------|
| AGE                 | 0.9553    | 0.9607 | 0.9580   | 356     |
| DATE                | 0.9609    | 0.9791 | 0.9699   | 1103    |
| GENDER              | 0.9487    | 0.9453 | 0.9470   | 274     |
| JOB                 | 0.8163    | 0.6107 | 0.6987   | 131     |
| LOCATION            | 0.9177    | 0.9322 | 0.9249   | 2727    |
| NAME                | 0.8902    | 0.8280 | 0.8579   | 186     |
| ORGANIZATION        | 0.8725    | 0.7949 | 0.8319   | 551     |
| PATIENT_ID          | 0.9573    | 0.9890 | 0.9729   | 1269    |
| SYMPTOM_AND_DISEASE | 0.8201    | 0.7794 | 0.7992   | 766     |
| TRANSPORTATION      | 0.9655    | 0.9655 | 0.9655   | 87      |



## 📈 Detailed Results by (Syllable Model)

| Entity              | Precision | Recall | F1-score | Support |
|---------------------|-----------|--------|----------|---------|
| AGE                 | 0.9413    | 0.9656 | 0.9533   | 349     |
| DATE                | 0.9593    | 0.9837 | 0.9713   | 1102    |
| GENDER              | 0.9403    | 0.9299 | 0.9351   | 271     |
| JOB                 | 0.8529    | 0.6744 | 0.7532   | 129     |
| LOCATION            | 0.8978    | 0.9213 | 0.9094   | 2707    |
| NAME                | 0.8674    | 0.8441 | 0.8556   | 186     |
| ORGANIZATION        | 0.8426    | 0.8303 | 0.8364   | 548     |
| PATIENT_ID          | 0.9672    | 0.9786 | 0.9729   | 1264    |
| SYMPTOM_AND_DISEASE | 0.7947    | 0.7906 | 0.7927   | 764     |
| TRANSPORTATION      | 0.9213    | 0.9425 | 0.9318   | 87      |

---

## 📊 Overall Performance

### ⚙️ Words Model
|               | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| micro avg     | 0.9201    | 0.9169 | 0.9185   | 7450    |
| macro avg     | 0.9104    | 0.8785 | 0.8926   | 7450    |
| weighted avg  | 0.9185    | 0.9169 | 0.9171   | 7450    |


### ⚙️ Syllable Model
|               | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| micro avg     | 0.9070    | 0.9166 | 0.9118   | 7407    |
| macro avg     | 0.8985    | 0.8861 | 0.8912   | 7407    |
| weighted avg  | 0.9064    | 0.9166 | 0.9112   | 7407    |

---

## 🔍 Observations

- **Strong performance:** DATE, AGE, PATIENT_ID.  
- **Good performance:** LOCATION, GENDER.
- **Lower performance:**
  - JOB (class imbalance, sparse samples).
  - SYMPTOM_AND_DISEASE (complex multi-token / multi-span entities).

---

## 🔍 Key Findings
- Word-level model performs well on structured entities like DATE and PATIENT_ID.
- Syllable-level representation is expected to improve:
  - Out-of-vocabulary handling.
  - Vietnamese word segmentation robustness.
- Combining both representations (future work) may further improve performance.

---

## 🚀 Future Work
- Hybrid Word + Syllable embedding fusion.
- Transformer-based NER (PhoBERT / XLM-R).
- Multi-task learning with POS tagging.
- Data augmentation for rare entities.

---

## 🚀 How to Use

```python
import json

with open("word2id.json", "r", encoding="utf-8") as f:
    word2id = json.load(f)

with open("char2id.json", "r", encoding="utf-8") as f:
    char2id = json.load(f)

with open("label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("hf_model/id2label.json", "w", encoding="utf-8") as f:
    json.dump(id2label, f, ensure_ascii=False, indent=2)
    
id2label = {int(k): v for k, v in label2id.items()}
"""

In [267]:
import os

os.makedirs("hf_model", exist_ok=True)

with open("hf_model/README.md", "w", encoding="utf-8") as f:
    f.write(README_text)

In [268]:
with open("hf_model/README.md", "r", encoding="utf-8") as f:
    print(f.read())


---
language:
  - vi.
tags:
  - ner.
  - medical.
  - covid-19.
  - bilstm-cnn-crf.
  - pytorch.
  - token-classification.
metrics:
  - precision.
  - recall.
  - f1.
---

# 🇻🇳 BiLSTM-CNN-CRF for Vietnamese COVID-19 NER

## 📌 Overview

This repository provides a **Named Entity Recognition (NER)** model for Vietnamese text in the **COVID-19 medical domain**.

The model is built using a hybrid deep learning architecture: **BiLSTM-CNN-CRF**, which captures both contextual and morphological features.

It is trained on the **PhoNER_COVID19 dataset** for extracting structured medical information from clinical and COVID-related texts.

---

## 🧩 Motivation

Vietnamese NER suffers from ambiguity in word segmentation. To address this, we explore two granularities:

- **Word-level representation**: captures semantic context.
- **Syllable-level representation**: better handles Vietnamese tokenization ambiguity and out-of-vocabulary words.

---

## 🧠 Model Architecture

We propose two variants of

In [269]:
import json
import os

os.makedirs("hf_model/words", exist_ok=True)

# word2id
with open("hf_model/words/word2id.json", "w", encoding="utf-8") as f:
    json.dump(word2id_words, f, ensure_ascii=False, indent=2)

# char2id
with open("hf_model/words/char2id.json", "w", encoding="utf-8") as f:
    json.dump(char2id_words, f, ensure_ascii=False, indent=2)

# label2id
with open("hf_model/words/label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id_words, f, ensure_ascii=False, indent=2)

# id2label (QUAN TRỌNG cho inference)
with open("hf_model/words/id2label.json", "w", encoding="utf-8") as f:
    json.dump(id2label_words, f, ensure_ascii=False, indent=2)

In [270]:
os.makedirs("hf_model/syllable", exist_ok=True)

# word2id (syllable vocab)
with open("hf_model/syllable/word2id.json", "w", encoding="utf-8") as f:
    json.dump(word2id_syllable, f, ensure_ascii=False, indent=2)

# char2id
with open("hf_model/syllable/char2id.json", "w", encoding="utf-8") as f:
    json.dump(char2id_syllable, f, ensure_ascii=False, indent=2)

# label2id
with open("hf_model/syllable/label2id.json", "w", encoding="utf-8") as f:
    json.dump(label2id_syllable, f, ensure_ascii=False, indent=2)

# id2label
with open("hf_model/syllable/id2label.json", "w", encoding="utf-8") as f:
    json.dump(id2label_syllable, f, ensure_ascii=False, indent=2)

In [271]:
import shutil

shutil.copy("/kaggle/working/best_words_model.pt", "hf_model/words/best_words_model.pt")

'hf_model/words/best_words_model.pt'

In [272]:
import shutil

shutil.copy("/kaggle/working/best_syllable_model.pt", "hf_model/syllable/best_syllable_model.pt")

'hf_model/syllable/best_syllable_model.pt'

In [273]:
import os

for f in os.listdir("hf_model/words"):
    print(os.path.join("hf_model/words", f))

hf_model/words/best_words_model.pt
hf_model/words/label2id.json
hf_model/words/char2id.json
hf_model/words/word2id.json
hf_model/words/id2label.json


In [277]:
from huggingface_hub import upload_file

repo_id = "ttnuizrachel/BiLSTM-CNN-CRF"

files = [
    # Words Model
    ("hf_model/words/best_words_model.pt", "words/best_model.pt"),
    ("hf_model/words/word2id.json", "words/word2id.json"),
    ("hf_model/words/char2id.json", "words/char2id.json"),
    ("hf_model/words/label2id.json", "words/label2id.json"),
    ("hf_model/words/id2label.json", "words/id2label.json"),

    # Syllable
    ("hf_model/syllable/best_syllable_model.pt", "syllable/best_model.pt"),
    ("hf_model/syllable/word2id.json", "syllable/word2id.json"),
    ("hf_model/syllable/char2id.json", "syllable/char2id.json"),
    ("hf_model/syllable/label2id.json", "syllable/label2id.json"),
    ("hf_model/syllable/id2label.json", "syllable/id2label.json"),

    # README
    ("hf_model/README.md", "README.md"),
]

for local, remote in files:
    upload_file(
        path_or_fileobj=local,
        path_in_repo=remote,
        repo_id=repo_id
    )

print("Upload thành công!")

Upload thành công!
